# New Features Showcase: v3.10.0 and v3.11.0

This notebook demonstrates modules and capabilities introduced in **siege_utilities**
versions 3.10.0 and 3.11.0. Each section covers one feature area with imports,
sample usage, and explanatory notes.

All examples use `try/except` guards for optional dependencies so the notebook
renders cleanly even when not every extra is installed.

### v3.10.0 features

| Feature | Module | Key API |
|---------|--------|---------|
| Vector/raster chart export | `reporting.chart_generator` | `save_figure_as_vector()` |
| GeometryPayload | `geo.spatial_runtime` | `GeometryPayload`, `encode_geometry()`, `decode_geometry()` |
| Design kit export | `reporting.examples.google_analytics_report_example` | `export_design_kit()` |

### v3.11.0 features

| Feature | Module | Key API |
|---------|--------|---------|
| 3D map rendering | `reporting.map_3d` | `ThreeDMapRenderer`, `create_3d_hexbin()` |
| H3 hexagonal indexing | `geo.h3_utils` | `h3_index_points()`, `h3_spatial_join()` |
| Isochrone providers | `geo.isochrones` | `IsochroneProvider`, `get_provider()` |
| Boundary providers | `geo.boundary_providers` | `BoundaryProvider`, `resolve_boundary_provider()` |
| Engine-agnostic DataFrames | `data.dataframe_engine` | `Engine`, `get_engine()` |
| IDML InDesign export | `reporting.idml_export` | `IDMLExporter`, `export_report_idml()` |

In [ ]:
import sys
from pathlib import Path

# Ensure siege_utilities is importable from the repo root
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

su.log_info("NB20 — New Features v3.10/v3.11 initialized.")

---
## 1. Vector / Raster Chart Export (v3.10.0)

The `ChartGenerator` gained `save_figure_as_vector()`, which exports any
matplotlib figure to **SVG**, **EPS**, or **PDF** format. This is critical for
designer handoff — InDesign and Illustrator can import SVG/EPS directly at any
resolution.

### When to use

- **Print reports**: Export SVG/EPS for infinite-resolution print output.
- **Design handoff**: Give designers editable vector charts they can restyle.
- **Archival**: PDF vector copies preserve chart fidelity indefinitely.

In [ ]:
import tempfile

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    su.log_info("matplotlib not installed — skipping vector export demo.")

if HAS_MATPLOTLIB:
    from siege_utilities.reporting.chart_generator import ChartGenerator

    cg = ChartGenerator()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Q1", "Q2", "Q3", "Q4"], [120, 180, 150, 210], color="#4a90d9")
    ax.set_title("Quarterly Revenue")
    ax.set_ylabel("Revenue ($k)")

    with tempfile.TemporaryDirectory() as tmpdir:
        svg_path = cg.save_figure_as_vector(fig, f"{tmpdir}/revenue", fmt="svg")
        eps_path = cg.save_figure_as_vector(fig, f"{tmpdir}/revenue", fmt="eps")
        pdf_path = cg.save_figure_as_vector(fig, f"{tmpdir}/revenue", fmt="pdf")

        print("Exported vector files:")
        for p in [svg_path, eps_path, pdf_path]:
            if p and p.exists():
                print(f"  {p.name:<20} {p.stat().st_size:>8,} bytes")

    plt.close(fig)
    print("\nVector export complete.")

---
## 2. GeometryPayload — Spark-Safe Geometry Transport (v3.10.0)

The `GeometryPayload` dataclass in `geo.spatial_runtime` provides a
Spark-safe container for transporting shapely geometries through
DataFrames, Spark rows, and serialisation boundaries.

Key codec functions:

- `encode_geometry(geom)` — shapely geometry to `GeometryPayload`
- `decode_geometry(payload)` — `GeometryPayload` back to shapely
- `payload_to_spark_row(payload)` — payload to a plain `dict` (Spark-safe)
- `spark_row_to_payload(row)` — plain `dict` back to payload

### When to use

- Passing geometries through Spark UDFs or Delta Lake columns.
- Serialising geometries to JSON/Parquet without losing CRS metadata.
- Round-tripping geometries across process boundaries.

In [ ]:
from siege_utilities.geo.spatial_runtime import (
    GeometryPayload,
    encode_geometry,
    decode_geometry,
    payload_to_spark_row,
    spark_row_to_payload,
    resolve_spatial_runtime_plan,
)

try:
    from shapely.geometry import Point, Polygon
    HAS_SHAPELY = True
except ImportError:
    HAS_SHAPELY = False
    su.log_info("shapely not installed — showing GeometryPayload structure only.")

if HAS_SHAPELY:
    pt = Point(-77.0369, 38.9072)  # Washington DC
    payload = encode_geometry(pt)

    print("Encoded GeometryPayload:")
    print(f"  geometry_type : {payload.geometry_type}")
    print(f"  crs           : {payload.crs}")
    print(f"  WKT           : {payload.geometry_wkt}")
    print(f"  WKB (hex)     : {payload.geometry_wkb[:20]}...")
    print(f"  GeoJSON       : {payload.geometry_geojson}")

    spark_row = payload_to_spark_row(payload)
    print("\nSpark-safe row keys:", list(spark_row.keys()))

    restored_payload = spark_row_to_payload(spark_row)
    restored_geom = decode_geometry(restored_payload)
    print(f"\nRound-trip result: {restored_geom.wkt}")
    print(f"  Match original: {pt.equals(restored_geom)}")
else:
    print("GeometryPayload fields:")
    empty = GeometryPayload()
    for field_name in ["geometry_wkt", "geometry_wkb", "geometry_geojson", "crs", "geometry_type"]:
        print(f"  {field_name:<20} = {getattr(empty, field_name)!r}")

In [ ]:
plan = resolve_spatial_runtime_plan()

print("Spatial Runtime Plan:")
print(f"  runtime                : {plan.runtime}")
print(f"  native_spatial_available: {plan.native_spatial_available}")
print(f"  sedona_available       : {plan.sedona_available}")
print(f"  loader_order           : {plan.loader_order}")
print(f"  reason                 : {plan.reason}")

plan_dbr = resolve_spatial_runtime_plan(databricks_runtime_version="17.1.x-scala2.12")
print(f"\nSimulated Databricks 17.1:")
print(f"  runtime     : {plan_dbr.runtime}")
print(f"  loader_order: {plan_dbr.loader_order}")

---
## 3. Design Kit Export (v3.10.0)

The `export_design_kit()` function produces a complete folder structure
with separated assets for InDesign/Illustrator handoff:

```
output_dir/
    charts/          SVG vector chart files
    charts-png/      High-resolution PNG chart files (300 DPI)
    tables/          CSV exports of all data tables
    text/            report_text.md with narrative sections
    metadata.yaml    Client name, dates, KPIs, summary stats
```

### When to use

- Handing off analytics results to a graphic designer.
- Producing multi-format deliverables from a single data dict.
- Archiving reports with both vector and raster assets.

In [ ]:
try:
    from siege_utilities.reporting.examples.google_analytics_report_example import (
        export_design_kit,
        generate_sample_ga_data,
    )
    HAS_DESIGN_KIT = True
except ImportError as e:
    HAS_DESIGN_KIT = False
    su.log_info(f"Design kit import failed ({e}) — showing API signature only.")

if HAS_DESIGN_KIT:
    ga_data = generate_sample_ga_data()
    print("Sample GA data keys:", list(ga_data.keys()))
    print(f"  Total users   : {ga_data['totals']['users']:,}")
    print(f"  Total sessions: {ga_data['totals']['sessions']:,}")

    with tempfile.TemporaryDirectory() as tmpdir:
        kit_dir = f"{tmpdir}/design_kit"
        success = export_design_kit(
            ga_data,
            output_dir=kit_dir,
            client_name="Demo Client",
            prepared_by="Siege Analytics",
        )
        print(f"\nDesign kit export success: {success}")

        if success:
            kit_path = Path(kit_dir)
            for item in sorted(kit_path.rglob("*")):
                rel = item.relative_to(kit_path)
                if item.is_file():
                    print(f"  {rel}  ({item.stat().st_size:,} bytes)")
                else:
                    print(f"  {rel}/")
else:
    print("export_design_kit(ga_data, output_dir, client_name, prepared_by, branding_key)")
    print("  -> Produces charts/, charts-png/, tables/, text/, metadata.yaml")

---
## 4. 3D Map Rendering with pydeck (v3.11.0)

The `reporting.map_3d` module provides `ThreeDMapRenderer` for creating
interactive 3D maps using pydeck (deck.gl). It supports:

- **Hexagonal binning** — aggregate point data into 3D hex columns
- **Column layers** — one column per data point, height proportional to value
- **3D choropleths** — extruded polygons from a GeoDataFrame
- **HTML export** and **in-notebook rendering**

### When to use

- Visualising spatial density (donations, voters, events) in 3D.
- Creating interactive map widgets in Jupyter or Databricks notebooks.
- Exporting standalone HTML maps for stakeholder review.

In [ ]:
try:
    from siege_utilities.reporting.map_3d import (
        ThreeDMapRenderer,
        create_3d_hexbin,
        create_3d_columns,
        MAP_STYLES,
        PYDECK_AVAILABLE,
    )
    HAS_MAP3D = True
except ImportError as e:
    HAS_MAP3D = False
    su.log_info(f"map_3d import failed ({e}).")

if HAS_MAP3D:
    print("Available map styles:")
    for name, url in MAP_STYLES.items():
        print(f"  {name:<12} {url}")

    print(f"\npydeck installed: {PYDECK_AVAILABLE}")

    if PYDECK_AVAILABLE:
        import pandas as pd
        import numpy as np

        rng = np.random.default_rng(42)
        n = 500
        sample_df = pd.DataFrame({
            "latitude": 38.9 + rng.normal(0, 0.05, n),
            "longitude": -77.0 + rng.normal(0, 0.05, n),
            "value": rng.integers(100, 5000, n),
        })
        print(f"\nSample data: {len(sample_df)} points")
        print(sample_df.head())

        renderer = ThreeDMapRenderer(map_style="dark")
        deck = renderer.create_hexagon_layer(
            sample_df, radius=500, elevation_scale=50, zoom=11,
        )
        print(f"\nHexbin deck created: {type(deck).__name__}")

        deck_cols = renderer.create_column_layer(
            sample_df, value_col="value", radius=200,
            elevation_scale=1, color=[30, 144, 255, 180], zoom=12,
        )
        print(f"Column deck created: {type(deck_cols).__name__}")

        with tempfile.TemporaryDirectory() as tmpdir:
            html_path = renderer.render_to_html(deck, f"{tmpdir}/hexbin_map.html")
            print(f"\nHTML export: {html_path.name} ({html_path.stat().st_size:,} bytes)")
    else:
        print("\nInstall pydeck to see interactive 3D maps:")
        print("  pip install 'siege-utilities[3d]'")
else:
    print("map_3d module not available.")

---
## 5. H3 Hexagonal Spatial Indexing (v3.11.0)

The `geo.h3_utils` module provides lightweight spatial join approximation
using Uber's H3 hexagonal indexing system. This is a "geo-lite" fallback
when full GeoPandas `sjoin` operations are too expensive.

Key functions:

- `h3_index_points(df, lat_col, lon_col, resolution)` — assign H3 hex IDs
- `h3_index_polygon(geometry, resolution)` — get hex IDs covering a polygon
- `h3_spatial_join(points_df, polygons_gdf, ...)` — approximate point-in-polygon
- `h3_hex_to_boundary(hex_id)` — get boundary coordinates for a hex cell
- `h3_resolution_for_area(target_km2)` — suggest resolution for a target area

### When to use

- Millions of point-in-polygon joins where GeoPandas is too slow.
- Spatial aggregation at multiple resolutions.
- Pre-indexing data for fast lookups in Spark or DuckDB.

In [ ]:
try:
    from siege_utilities.geo.h3_utils import (
        h3_index_points,
        h3_index_polygon,
        h3_spatial_join,
        h3_hex_to_boundary,
        h3_resolution_for_area,
        H3_AVAILABLE,
        _H3_RESOLUTION_AREA_KM2,
    )
    HAS_H3_MODULE = True
except ImportError as e:
    HAS_H3_MODULE = False
    su.log_info(f"h3_utils import failed ({e}).")

if HAS_H3_MODULE:
    print("H3 Resolution Reference (approximate hex areas):")
    print(f"{'Res':>4}  {'Area (km2)':>14}  {'~Description'}")
    print("-" * 50)
    descriptions = {
        0: "Continent", 1: "Large country", 2: "Large state",
        3: "US state", 4: "Large county", 5: "City metro",
        6: "Small city", 7: "Large neighbourhood", 8: "Neighbourhood",
        9: "City block", 10: "Sub-block", 11: "Building",
        12: "Small building", 13: "Room", 14: "Desk", 15: "Tiny",
    }
    for res, area in _H3_RESOLUTION_AREA_KM2.items():
        print(f"{res:>4}  {area:>14,.3f}  {descriptions.get(res, '')}")

    print(f"\nh3 library available: {H3_AVAILABLE}")
else:
    print("h3_utils module not available.")

In [ ]:
if HAS_H3_MODULE and H3_AVAILABLE:
    import pandas as pd

    points = pd.DataFrame({
        "latitude": [38.9072, 38.8977, 38.8895, 40.7128, 40.7580],
        "longitude": [-77.0369, -77.0365, -77.0353, -74.0060, -73.9855],
        "label": ["White House", "Washington Monument", "Lincoln Memorial",
                  "NYC City Hall", "Times Square"],
    })

    hex_ids = h3_index_points(points, "latitude", "longitude", resolution=8)
    print("H3-indexed points (resolution 8):")
    result = points.copy()
    result["h3_index"] = hex_ids
    print(result[["label", "h3_index"]].to_string(index=False))

    first_hex = hex_ids.iloc[0]
    boundary = h3_hex_to_boundary(first_hex)
    print(f"\nBoundary vertices for {first_hex}: {len(boundary)} points")
    for lat, lng in boundary[:3]:
        print(f"  ({lat:.6f}, {lng:.6f})")
    print("  ...")

    print("\nResolution suggestions:")
    for target_km2 in [1.0, 10.0, 100.0, 1000.0]:
        res = h3_resolution_for_area(target_km2)
        actual = _H3_RESOLUTION_AREA_KM2[res]
        print(f"  Target {target_km2:>7,.1f} km2 -> resolution {res} (actual {actual:,.3f} km2)")

elif HAS_H3_MODULE:
    print("h3 library not installed. Install with:")
    print("  pip install 'siege-utilities[h3]'")

---
## 6. Isochrone Providers (v3.11.0)

The `geo.isochrones` module provides both a function-based API and an
OOP provider architecture for fetching isochrone (travel-time contour)
polygons from routing services.

### Provider hierarchy

- `IsochroneProvider` (ABC) — abstract base with `fetch()`, `validate_config()`
- `OpenRouteServiceProvider` — wraps the ORS API (hosted or self-hosted)
- `ValhallaProvider` — wraps a Valhalla routing server
- `get_provider(name, **kwargs)` — factory function

### When to use

- Determining service areas (e.g., 15-minute drive from an office).
- Voter outreach planning: which voters are reachable within N minutes.
- Site selection: comparing isochrone coverage across candidate locations.

In [ ]:
from siege_utilities.geo.isochrones import (
    IsochroneProvider,
    OpenRouteServiceProvider,
    ValhallaProvider,
    get_provider,
    build_isochrone_request,
    IsochroneRequest,
    DEFAULT_ORS_BASE_URL,
    DEFAULT_VALHALLA_BASE_URL,
)

print("Isochrone Provider Hierarchy:")
print(f"  ABC: {IsochroneProvider.__name__}")
print(f"  Concrete: {OpenRouteServiceProvider.__name__}, {ValhallaProvider.__name__}")

ors = get_provider("ors", api_key="demo-key")
valhalla = get_provider("valhalla", base_url="http://valhalla.elect.info:8002")

print(f"\nORS provider:")
print(f"  name     : {ors.provider_name}")
print(f"  base_url : {ors.base_url}")
print(f"  valid    : {ors.validate_config()}")

print(f"\nValhalla provider:")
print(f"  name     : {valhalla.provider_name}")
print(f"  base_url : {valhalla.base_url}")
print(f"  valid    : {valhalla.validate_config()}")

In [ ]:
request_ors = build_isochrone_request(
    latitude=38.9072, longitude=-77.0369,
    travel_time_minutes=15, provider="ors",
    profile="driving-car", api_key="demo-key",
)

print("ORS Isochrone Request:")
for key, val in request_ors.items():
    print(f"  {key:<10} : {val}")

print()

request_val = build_isochrone_request(
    latitude=38.9072, longitude=-77.0369,
    travel_time_minutes=15, provider="valhalla",
    profile="driving-car",
    base_url="http://valhalla.elect.info:8002",
)

print("Valhalla Isochrone Request:")
for key, val in request_val.items():
    print(f"  {key:<10} : {val}")

---
## 7. Boundary Providers (v3.11.0)

The `geo.boundary_providers` module provides a pluggable architecture for
fetching administrative boundary geometries from different sources:

- `BoundaryProvider` (ABC) — with `get_boundary()`, `list_levels()`, `is_available()`
- `CensusTIGERProvider` — US Census TIGER/Line shapefiles
- `GADMProvider` — international boundaries via GADM
- `resolve_boundary_provider(country)` — factory (TIGER for US, GADM otherwise)

### When to use

- Building geographic base layers for choropleths.
- Supporting both US and international campaigns from a single API.
- Fetching county, tract, or admin boundaries on demand.

In [ ]:
from siege_utilities.geo.boundary_providers import (
    BoundaryProvider,
    CensusTIGERProvider,
    GADMProvider,
    resolve_boundary_provider,
)

us_provider = resolve_boundary_provider("US")
de_provider = resolve_boundary_provider("DE")
pr_provider = resolve_boundary_provider("PR")

print("Provider Resolution:")
for country, prov in [("US", us_provider), ("DE", de_provider), ("PR", pr_provider)]:
    print(f"  {country} -> {prov.provider_name} (available: {prov.is_available()})")

tiger = CensusTIGERProvider()
print(f"\nCensus TIGER levels: {tiger.list_levels()}")

gadm = GADMProvider()
print(f"GADM levels: {gadm.list_levels()}")
print(f"GADM available (geopandas): {gadm.is_available()}")

---
## 8. Engine-Agnostic DataFrame Operations (v3.11.0)

The `data.dataframe_engine` module provides a thin abstraction so the same
analytical operations can run on pandas, DuckDB, Spark, or PostGIS.

| Engine | Class | Requires |
|--------|-------|----------|
| `pandas` | `PandasEngine` | pandas (always available) |
| `duckdb` | `DuckDBEngine` | duckdb |
| `spark` | `SparkEngine` | pyspark |
| `postgis` | `PostGISEngine` | sqlalchemy + geopandas |

### When to use

- Writing analysis code that should work in both local (pandas) and
  cluster (Spark) environments.
- Switching between DuckDB and PostGIS without rewriting queries.
- Testing pipeline logic locally with pandas, deploying on Spark.

In [ ]:
from siege_utilities.data.dataframe_engine import (
    Engine,
    get_engine,
    DataFrameEngine,
    PandasEngine,
)

print("Available engine back-ends:")
for e in Engine:
    print(f"  {e.value}")

engine = get_engine("pandas")
print(f"\nActive engine: {engine.name}")
print(f"  type: {type(engine).__name__}")

In [ ]:
import pandas as pd

engine = get_engine(Engine.PANDAS)

df = pd.DataFrame({
    "state": ["VA", "VA", "MD", "MD", "DC", "DC"],
    "county": ["Fairfax", "Arlington", "Montgomery", "PG", "Ward 1", "Ward 2"],
    "population": [1150309, 238643, 1062061, 967201, 78000, 82000],
    "donations": [4500, 2800, 3200, 1900, 1200, 1100],
})

state_totals = engine.groupby_agg(df, ["state"], {"population": "sum", "donations": "sum"})
print("Group by state (sum):")
print(state_totals.to_string(index=False))

large = engine.filter(df, df["population"] > 500000)
print(f"\nCounties with pop > 500k: {len(large)}")
print(large[["county", "population"]].to_string(index=False))

state_names = pd.DataFrame({
    "state": ["VA", "MD", "DC"],
    "full_name": ["Virginia", "Maryland", "District of Columbia"],
})
joined = engine.join(state_totals, state_names, on="state", how="left")
print("\nJoined with state names:")
print(joined.to_string(index=False))

In [ ]:
print("Engine availability check:")
for eng_enum in Engine:
    try:
        e = get_engine(eng_enum)
        print(f"  {eng_enum.value:<10} AVAILABLE  ({type(e).__name__})")
    except (ImportError, ValueError) as exc:
        print(f"  {eng_enum.value:<10} not installed  ({exc.__class__.__name__})")

---
## 9. IDML InDesign Export (v3.11.0)

The `reporting.idml_export` module enables programmatic creation of
Adobe InDesign IDML files. IDML is a ZIP archive containing XML files
that InDesign can open directly.

### Two modes

1. **Template mode** (requires `simpleidml`): Load an `.idml` template
   and replace placeholders.
2. **Manual mode** (zero dependencies): Build the IDML ZIP from scratch.

### When to use

- Producing InDesign-ready reports from analytics pipelines.
- Letting designers customise report layouts while preserving data.
- Generating multi-format output (PDF + PPTX + IDML) from one data source.

In [ ]:
from siege_utilities.reporting.idml_export import (
    IDMLExporter,
    export_report_idml,
    SIMPLEIDML_AVAILABLE,
)

print(f"simpleidml available: {SIMPLEIDML_AVAILABLE}")
print("  (Manual ZIP builder works without it.)\n")

exporter = IDMLExporter(page_width=612, page_height=792)

exporter.add_text_frame(
    "Campaign Finance Report", style_name="Title",
    x=72, y=72, width=468, height=36,
)

exporter.add_text_frame(
    "Q4 2025 Summary", style_name="Subtitle",
    x=72, y=114, width=468, height=24,
)

exporter.add_table(
    headers=["Candidate", "Raised", "Spent", "COH"],
    rows=[
        ["Smith, J.", "$1,250,000", "$980,000", "$270,000"],
        ["Jones, A.", "$890,000", "$750,000", "$140,000"],
        ["Williams, R.", "$2,100,000", "$1,800,000", "$300,000"],
    ],
    x=72, y=160, width=468, height=100,
)

exporter.add_image_frame(
    "/path/to/spending_chart.svg",
    x=72, y=280, width=468, height=300,
)

with tempfile.TemporaryDirectory() as tmpdir:
    output = exporter.save(f"{tmpdir}/campaign_report")
    output_path = Path(output)
    print(f"IDML file: {output_path.name}")
    print(f"  Size: {output_path.stat().st_size:,} bytes")

    import zipfile
    with zipfile.ZipFile(output) as zf:
        print(f"  ZIP entries: {len(zf.namelist())}")
        for name in sorted(zf.namelist()):
            info = zf.getinfo(name)
            print(f"    {name:<45} {info.file_size:>6} bytes")

In [ ]:
sample_ga_data = {
    "date_range": {"start": "2025-10-01", "end": "2025-12-31"},
    "totals": {"users": 45200, "sessions": 62800, "pageviews": 198500, "avg_bounce_rate": 42.3},
    "changes": {},
    "traffic_sources": [
        {"source": "google", "medium": "organic", "sessions": 28000, "users": 22000, "bounce_rate": 38.5},
        {"source": "direct", "medium": "none", "sessions": 15000, "users": 12000, "bounce_rate": 45.2},
    ],
    "top_pages": [
        {"page": "/", "pageviews": 52000, "unique_views": 41000, "avg_time": 65.3, "bounce_rate": 35.0},
        {"page": "/donate", "pageviews": 18000, "unique_views": 15000, "avg_time": 120.5, "bounce_rate": 22.0},
    ],
    "devices": [
        {"device": "mobile", "sessions": 35000, "bounce_rate": 48.0},
        {"device": "desktop", "sessions": 24000, "bounce_rate": 35.0},
    ],
}

with tempfile.TemporaryDirectory() as tmpdir:
    result = export_report_idml(
        sample_ga_data, output_path=f"{tmpdir}/ga_report",
        title="Google Analytics Report",
    )
    result_path = Path(result)
    print(f"GA report IDML: {result_path.name} ({result_path.stat().st_size:,} bytes)")

---
## Summary

### v3.10.0

1. **Vector/Raster Chart Export** — `save_figure_as_vector()` for SVG/EPS/PDF.
2. **GeometryPayload** — Spark-safe geometry transport with WKT/WKB/GeoJSON codecs.
3. **Design Kit Export** — `export_design_kit()` for separated assets.

### v3.11.0

4. **3D Map Rendering** — `ThreeDMapRenderer` with pydeck layers.
5. **H3 Hexagonal Indexing** — Lightweight spatial joins via H3.
6. **Isochrone Providers** — ABC + ORS/Valhalla implementations.
7. **Boundary Providers** — Census TIGER (US) + GADM (international).
8. **Engine-Agnostic DataFrames** — Uniform API across pandas/DuckDB/Spark/PostGIS.
9. **IDML InDesign Export** — Programmatic IDML creation.

### Key Files

| File | Purpose |
|------|---------|
| `reporting/chart_generator.py` | Vector/raster export |
| `geo/spatial_runtime.py` | GeometryPayload |
| `reporting/examples/google_analytics_report_example.py` | Design kit |
| `reporting/map_3d.py` | 3D map rendering |
| `geo/h3_utils.py` | H3 hexagonal indexing |
| `geo/isochrones.py` | Isochrone providers |
| `geo/boundary_providers.py` | Boundary providers |
| `data/dataframe_engine.py` | Engine-agnostic DFs |
| `reporting/idml_export.py` | IDML InDesign export |